This file used to transform videos to latent, which will be ground truth for Seq2seq

## CONFIGURATION

In [2]:
CONFIG = {
    "video_file_path": r"D:\00 download\工作区\final\output_videos_v2_fixed", # "data/videos"
    "save_latent_path": "data/metadata/videos_latents.pt",
    "num_videos": 250, 

    "model_name": "CompVis/stable-diffusion-v1-4",
    "video_height": 1080,      # 视频高度
    "video_width": 1920,       # 视频宽度
    "desample_rate": 3.75,     # 下采样率
    "video_frame_rate": 5,     # 视频帧率
    "video_num_frames": 12,    # 每个视频的帧数
    "batch_size": 5,           # 批处理大小

    "save_frames_path": "data/metadata/videos_frames.npy",
}


## Base settings

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Info]: Use {device} now!")


[Info]: Use cpu now!


## Load VAE model

In [8]:
from diffusers import AutoencoderKL

model_path = CONFIG["model_name"]
print(f"Loading VAE from '{model_path}'...")
vae = AutoencoderKL.from_pretrained(
    model_path,
    subfolder="vae",
    torch_dtype=torch.float16,
).to(device)
vae.eval()
print("VAE loaded successfully.")

Loading VAE from 'CompVis/stable-diffusion-v1-4'...
VAE loaded successfully.


## Extract frames from one video

In [4]:
import decord

decord.bridge.set_bridge('torch')

def extract_frames(video_path): 
    video_reader = decord.VideoReader(str(video_path), width=int(CONFIG["video_width"]/CONFIG["desample_rate"]), height=int(CONFIG["video_height"]/CONFIG["desample_rate"]))
    video_length = len(video_reader)

    # 计算采样索引
    sample_indices = list(range(0, video_length, CONFIG["video_frame_rate"]))[:CONFIG["video_num_frames"]]
    
    # 读取帧
    frames = video_reader.get_batch(sample_indices).to(device, dtype=torch.float16)

    return frames

## rearrange frames to fit the VAE

In [10]:
from einops import rearrange

def video2latent(frames, vae):
    with torch.no_grad():
        # change value from [0, 255] to [-1, 1]
        frames = (frames / 127.5) - 1.0

        # 调整维度以适应 VAE: (V, F, H, W, C) -> (V*F, C, H, W)
        pixel_values_flat = rearrange(frames, "v f h w c -> (v f) c h w")

        # 编码
        latents_dist = vae.encode(pixel_values_flat).latent_dist
        latents_flat = latents_dist.sample()

        # 缩放并恢复维度
        latents_flat = latents_flat * vae.config.scaling_factor
        latents = rearrange(latents_flat, "(v f) c h w -> v f c h w", f=CONFIG["video_num_frames"])

    return latents

## Start!

In [ ]:
from tqdm.auto import tqdm
import os

all_latents = []

for i in tqdm(range(int(CONFIG["num_videos"]/CONFIG["batch_size"])), desc="Encoding video batches"): 
    batch_path_list = [os.path.join(CONFIG["video_file_path"], f"{j+1}.mp4") for j in range(i*CONFIG["batch_size"], (i+1)*CONFIG["batch_size"])]
    batch_frames = [extract_frames(path) for path in batch_path_list]
    batch_frames = torch.stack(batch_frames)
    batch_latents = video2latent(batch_frames, vae).cpu()
    all_latents.append(batch_latents)

final_latents_tensor = torch.cat(all_latents, dim=0)
torch.save(final_latents_tensor, CONFIG["save_latent_path"])# (250, 12, 4, 36, 64)

## Save the frames data for [10_train_finetune_videodiffusion.ipynb](10_train_finetune_videodiffusion.ipynb)

In [6]:
import numpy as np
import os
from tqdm.auto import tqdm
from einops import rearrange

frames = []
for i in tqdm(range(int(CONFIG["num_videos"])), desc="Save frames data"):
    video_path = os.path.join(CONFIG["video_file_path"], f"{i+1}.mp4")
    frames.append(extract_frames(video_path))

frames = np.stack(frames)
frames = rearrange(frames, "v f h w c -> v f c h w")
print(frames.shape)
np.save(CONFIG["save_frames_path"], frames) # (60, 12, 3, 288, 512)

d:\app\miniconda\envs\eeg_test\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Save frames data: 100%|██████████| 250/250 [00:24<00:00, 10.23it/s]


(250, 12, 3, 288, 512)
